# Построение профилей клиентов и сортировка транзакций
В этом ноутбуке:
- загружаем `train_transaction.csv` и `train_identity.csv` (chunk-friendly);
- строим агрегированные профили по `card1`;
- встраиваем профильные признаки в `train_identity` и сохраняем;
- сортируем `train_transaction` по `card1` затем по времени и сохраняем.

In [13]:
import os
import pandas as pd
import numpy as np
from pathlib import Path

base_path = Path("/content/drive/MyDrive/ieee-db")

tx_path = base_path / "train_transaction.csv"
id_path = base_path / "train_identity.csv"

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

print("Базовый путь Google Drive:", base_path)
print("Файл транзакций найден:", tx_path.exists())
print("Файл identity найден:", id_path.exists())
if not tx_path.exists():
    print("\n Файлы не найдены! Проверьте содержимое папки:")
    if base_path.exists():
        print(os.listdir(base_path))
    else:
        print("Папка ieee-db не существует по указанному пути.")

Базовый путь Google Drive: /content/drive/MyDrive/ieee-db
Файл транзакций найден: True
Файл identity найден: True


In [14]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
if 'ensure_cols' not in globals():
    def ensure_cols(df):
        if 'TransactionAmt' not in df.columns and 'TransactionAMT' in df.columns:
            df = df.rename(columns={'TransactionAMT': 'TransactionAmt'})
        return df

# Разбиваем уникальные UID на train/val и сохраняем split-файлы чанками
# Выполняется перед агрегацией, чтобы избежать leakage.
print('Собираю уникальные UID (chunked)...')
clients = set()
cols_for_uid = ['card1','addr1','TransactionDT','D1']
for ch in pd.read_csv(tx_path, usecols=lambda c: c in cols_for_uid, chunksize=200000):
    ch = ensure_cols(ch)
    # гарантируем наличие колонок
    for c in cols_for_uid:
        if c not in ch.columns:
            ch[c] = np.nan
    # вычисляем D1n
    try:
        ch['D1n'] = np.floor(pd.to_numeric(ch['TransactionDT'], errors='coerce') / (24*3600)) - pd.to_numeric(ch['D1'], errors='coerce')
    except Exception:
        ch['D1n'] = np.nan
    ch['D1n'] = ch['D1n'].fillna(-9999).astype(int)
    ch['addr1'] = ch['addr1'].fillna('').astype(str)
    uids = (ch['card1'].astype(str) + '_' + ch['addr1'] + '_' + ch['D1n'].astype(str))
    clients.update(uids.dropna().unique().tolist())
clients = sorted(clients)
print('Всего UID:', len(clients))

import math
rng = np.random.RandomState(42)
perm = rng.permutation(len(clients))
train_frac = 0.8
train_n = int(len(clients) * train_frac)
train_clients = set([clients[i] for i in perm[:train_n]])
val_clients = set([clients[i] for i in perm[train_n:]])

train_clients_file = base_path / 'train_clients.txt'
val_clients_file = base_path / 'val_clients.txt'
with open(train_clients_file, 'w', encoding='utf8') as f:
    f.write('\n'.join(train_clients))
with open(val_clients_file, 'w', encoding='utf8') as f:
    f.write('\n'.join(val_clients))
print('Списки клиентов сохранены:', train_clients_file, val_clients_file)

train_tx_out = base_path / 'train_transactions_by_client.csv'
val_tx_out = base_path / 'val_transactions_by_client.csv'
first_train = True
first_val = True
for i, ch in enumerate(pd.read_csv(base_path / 'train_transaction.csv', chunksize=200000)):
    ch = ensure_cols(ch)
    for c in ['card1','addr1','TransactionDT','D1']:
        if c not in ch.columns:
            ch[c] = np.nan
    try:
        ch['D1n'] = np.floor(pd.to_numeric(ch['TransactionDT'], errors='coerce') / (24*3600)) - pd.to_numeric(ch['D1'], errors='coerce')
    except Exception:
        ch['D1n'] = np.nan
    ch['D1n'] = ch['D1n'].fillna(-9999).astype(int)
    ch['addr1'] = ch['addr1'].fillna('').astype(str)
    ch['client_id'] = ch['card1'].astype(str) + '_' + ch['addr1'] + '_' + ch['D1n'].astype(str)

    tr = ch[ch['client_id'].isin(train_clients)]
    va = ch[ch['client_id'].isin(val_clients)]
    if not tr.empty:
        tr.to_csv(train_tx_out, index=False, mode='w' if first_train else 'a', header=first_train)
        first_train = False
    if not va.empty:
        va.to_csv(val_tx_out, index=False, mode='w' if first_val else 'a', header=first_val)
        first_val = False
    if (i+1) % 5 == 0:
        print(f'Обработано чанков: {i+1}')

print('Файлы разбиения сохранены:')
print(' -', train_tx_out)
print(' -', val_tx_out)

# По умолчанию дальше агрегируется train split.
# Для агрегации валидации можно выбрать 'val' или None.
use_split_for = 'train'  # options: 'train' | 'val' | None
if use_split_for == 'train':
    tx_path = train_tx_out
elif use_split_for == 'val':
    tx_path = val_tx_out
print('Путь tx_path для последующей агрегации:', tx_path)


Собираю уникальные UID (chunked)...
Всего UID: 217850
Списки клиентов сохранены: /content/drive/MyDrive/ieee-db/train_clients.txt /content/drive/MyDrive/ieee-db/val_clients.txt
Файлы разбиения сохранены:
 - /content/drive/MyDrive/ieee-db/train_transactions_by_client.csv
 - /content/drive/MyDrive/ieee-db/val_transactions_by_client.csv
Путь tx_path для последующей агрегации: /content/drive/MyDrive/ieee-db/train_transactions_by_client.csv


In [6]:
# Безопасное чтение колонок с разными написаниями.
def ensure_cols(df):
    # стандартные имена колонок, которые мы будем использовать в коде
    # допускаем варианты TransactionAmt / TransactionAMT
    if "TransactionAmt" not in df.columns and "TransactionAMT" in df.columns:
        df = df.rename(columns={"TransactionAMT": "TransactionAmt"})
    return df

usecols_tx = [
    "TransactionID","card1","isFraud","TransactionDT","TransactionAmt",
    "ProductCD","card4","card6","addr1","addr2",
    # дополнительные колонки для формирования UID и фич
    "D1","D2","D4","D10","D15","C13","card2","dist1",
    "P_emaildomain","R_emaildomain"
]

sample = None
try:
    sample = pd.read_csv(tx_path, nrows=10)
    sample = ensure_cols(sample)
    print("Прочитано (sample) колонок:", list(sample.columns)[:40])
except Exception:
    print("Не удалось прочитать sample транзакций (файл может быть большой). Продолжим с чтением чанками.")

Прочитано (sample) колонок: ['TransactionID', 'isFraud', 'TransactionDT', 'TransactionAmt', 'ProductCD', 'card1', 'card2', 'card3', 'card4', 'card5', 'card6', 'addr1', 'addr2', 'dist1', 'dist2', 'P_emaildomain', 'R_emaildomain', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14', 'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9']


In [7]:
# Агрегация профилей по UID чанками.
# Медиана считается приближенно, потому что данные обрабатываются чанками.
from collections import defaultdict

agg_numeric = defaultdict(lambda: {"tx_count":0, "tx_amount_sum":0.0, "tx_amount_sq_sum":0.0, "fraud_count":0})
agg_nunique = {}
agg_modes = {}
dt_min = {}
dt_max = {}

chunk_iter = pd.read_csv(tx_path, usecols=lambda c: c in usecols_tx, chunksize=200000, iterator=True)
for i,chunk in enumerate(chunk_iter):
    chunk = ensure_cols(chunk)

    # гарантируем наличие колонок, которые мы используем для UID и агрегатов
    cols = ["card1","TransactionID","isFraud","TransactionDT","TransactionAmt","ProductCD","card4","card6","addr1","addr2","D1","D2","D4","D10","D15","C13","card2","dist1","P_emaildomain","R_emaildomain"]
    for col in cols:
        if col not in chunk.columns:
            chunk[col] = np.nan

    # вычисляем D1n и составной UID
    try:
        chunk["D1n"] = np.floor(pd.to_numeric(chunk["TransactionDT"], errors='coerce') / (24*3600)) - pd.to_numeric(chunk.get('D1', np.nan), errors='coerce')
    except Exception:
        chunk["D1n"] = np.nan
    chunk["D1n"] = chunk["D1n"].fillna(-9999).astype(int)
    chunk["addr1"] = chunk["addr1"].fillna("").astype(str)
    chunk["client_id"] = chunk["card1"].astype(str) + "_" + chunk["addr1"].astype(str) + "_" + chunk["D1n"].astype(str)

    # группировка по client_id (UID) в чанке
    grp = chunk.groupby("client_id")
    for client, g in grp:
        key = client
        cnt = len(g)
        amt_sum = g["TransactionAmt"].fillna(0).sum()
        frauds = int(g["isFraud"].fillna(0).sum())
        agg_numeric[key]["tx_count"] += cnt
        agg_numeric[key]["tx_amount_sum"] += float(amt_sum)
        agg_numeric[key]["fraud_count"] += frauds
        # для std/mean можем аккумулировать квадраты (Welford проще, но тут кол-ва большие)
        s = (g["TransactionAmt"].fillna(0)**2).sum()
        agg_numeric[key]["tx_amount_sq_sum"] += float(s)

        # уникальные счётчики (приближённо — собираем множества в памяти, осторожно)
        if key not in agg_nunique:
            agg_nunique[key] = {"productcd": set(), "addr1": set(), "addr2": set(), "device_type": set(), "card4": set(), "card6": set()}
        agg_nunique[key]["productcd"].update(g["ProductCD"].dropna().unique().tolist())
        agg_nunique[key]["addr1"].update(g["addr1"].dropna().unique().tolist())
        agg_nunique[key]["addr2"].update(g["addr2"].dropna().unique().tolist())
        agg_nunique[key]["card4"].update(g["card4"].dropna().unique().tolist())
        agg_nunique[key]["card6"].update(g["card6"].dropna().unique().tolist())

        # Сохраняем самые частые значения card4/card6 и их количества.
        if key not in agg_modes:
            agg_modes[key] = {"card4": defaultdict(int), "card6": defaultdict(int)}
        for v in g["card4"].dropna().astype(str):
            agg_modes[key]["card4"][v] += 1
        for v in g["card6"].dropna().astype(str):
            agg_modes[key]["card6"][v] += 1

        # dt min/max
        dtmin = g["TransactionDT"].min()
        dtmax = g["TransactionDT"].max()
        if key not in dt_min or (pd.notna(dtmin) and dtmin < dt_min[key]):
            dt_min[key] = dtmin
        if key not in dt_max or (pd.notna(dtmax) and dtmax > dt_max[key]):
            dt_max[key] = dtmax

    print(f"Обработан чанк {i+1}, клиентов в чанке: {grp.ngroups}")

print("Агрегация чанков завершена. Построим итогный DataFrame (может занять память).")

Обработан чанк 1, клиентов в чанке: 89050
Обработан чанк 2, клиентов в чанке: 83652
Обработан чанк 3, клиентов в чанке: 36553
Агрегация чанков завершена. Построим итогный DataFrame (может занять память).


In [8]:
# Собираем итоговый client_profile из накопленных структур.
rows = []
for client, vals in agg_numeric.items():
    tx_count = vals["tx_count"]
    amt_sum = vals["tx_amount_sum"]
    amt_sq = vals["tx_amount_sq_sum"]
    fraud_count = vals["fraud_count"] 
    # вычисляем mean/std из суммы и суммы квадратов (без необходимости держать все транзакции в памяти)
    mean_amt = amt_sum / tx_count if tx_count>0 else 0.0 
    var = (amt_sq / tx_count - mean_amt**2) if tx_count>0 else 0.0 # можем получить отрицательную дисперсию из-за численных ошибок, поэтому проверяем на >0
    std_amt = np.sqrt(var) if var>0 else 0.0

    prod_n = len(agg_nunique.get(client, {}).get("productcd", []))
    addr2_n = len(agg_nunique.get(client, {}).get("addr2", []))
    card4_mode = None
    card6_mode = None
    if client in agg_modes:
        if agg_modes[client]["card4"]:
            card4_mode = max(agg_modes[client]["card4"].items(), key=lambda x:x[1])[0]
        if agg_modes[client]["card6"]:
            card6_mode = max(agg_modes[client]["card6"].items(), key=lambda x:x[1])[0]

    dtmin = dt_min.get(client, np.nan)
    dtmax = dt_max.get(client, np.nan)
    span_days = (dtmax - dtmin) / (24*3600) if pd.notna(dtmin) and pd.notna(dtmax) else 0.0

    rows.append({
        "client_id": client,
        "tx_count_total": tx_count,
        "tx_amount_sum": amt_sum,
        "tx_amount_mean": mean_amt,
        "tx_amount_std": std_amt,
        "tx_count_fraud": fraud_count,
        "fraud_rate": fraud_count / tx_count if tx_count>0 else 0.0,
        "productcd_nunique": prod_n,
        "addr2_nunique": addr2_n,
        "card4_mode": card4_mode,
        "card6_mode": card6_mode,
        "tx_dt_span_days": span_days
    })

client_profile = pd.DataFrame(rows)
client_profile = client_profile.sort_values("tx_count_total", ascending=False).reset_index(drop=True)
print("Размер client_profile:", client_profile.shape)
display(client_profile.head(10))


Размер client_profile: (174280, 13)


,client_id,tx_count_total,tx_amount_sum,tx_amount_mean,tx_amount_std,tx_count_fraud,fraud_rate,productcd_nunique,addr1_nunique,addr2_nunique,card4_mode,card6_mode,tx_dt_span_days
0,15775_330.0_129,1414,146455.000,103.574965,19.805339,0,0.000000,1,1,1,mastercard,credit,52.046690
1,9500_126.0_-85,446,22679.450,50.850785,12.326475,0,0.000000,1,1,1,visa,debit,181.412650
2,8900_231.0_-60,232,18763.000,80.875000,54.565975,0,0.000000,1,1,1,visa,debit,124.641968
3,12616__-491,211,12175.860,57.705498,52.263435,37,0.175355,1,1,0,visa,credit,146.120174
4,12741_143.0_-202,196,10361.490,52.864745,33.098533,0,0.000000,1,1,1,visa,debit,83.494352
5,15885__22,174,7627.629,43.836948,30.208050,2,0.011494,1,1,0,visa,debit,160.622928
6,4121_476.0_-8,142,4977.500,35.052817,12.836966,0,0.000000,1,1,1,visa,credit,147.040347
7,15885__27,137,6863.688,50.099912,46.063975,8,0.058394,1,1,0,visa,debit,79.037917
8,15885__29,132,4974.311,37.684174,23.621101,2,0.015152,1,1,0,visa,debit,99.149433
9,15885__23,129,4785.675,37.098256,31.688476,0,0.000000,1,1,0,visa,debit,155.658194


In [9]:
# Объединяем identity с профилем клиента через TransactionID.
print("Читаем identity и объединяем с client_profile...")
identity = pd.read_csv(id_path)

tx_map = pd.read_csv(tx_path, usecols=lambda c: c in ["TransactionID","card1","addr1","TransactionDT","D1","P_emaildomain","R_emaildomain"]) if 'tx_path' in globals() else pd.read_csv(str(base_path / 'train_transaction.csv'), usecols=lambda c: c in ["TransactionID","card1","addr1","TransactionDT","D1","P_emaildomain","R_emaildomain"]) 
tx_map = ensure_cols(tx_map)
for c in ['TransactionDT','D1','card1','addr1']:
    if c not in tx_map.columns:
        tx_map[c] = np.nan
try:
    tx_map['D1n'] = np.floor(pd.to_numeric(tx_map['TransactionDT'], errors='coerce') / (24*3600)) - pd.to_numeric(tx_map['D1'], errors='coerce')
except Exception:
    tx_map['D1n'] = np.nan
tx_map['D1n'] = tx_map['D1n'].fillna(-9999).astype(int)
tx_map['addr1'] = tx_map['addr1'].fillna('').astype(str)
tx_map['client_id'] = tx_map['card1'].astype(str) + '_' + tx_map['addr1'] + '_' + tx_map['D1n'].astype(str)

identity_profile = identity.merge(tx_map[['TransactionID','client_id']], on="TransactionID", how="left")
identity_profile = identity_profile.merge(client_profile, on="client_id", how="left")

out_identity_profile = base_path / "train_identity_profiles.csv"
identity_profile.to_csv(out_identity_profile, index=False)
print("Сохранено:", out_identity_profile)
display(identity_profile.head(5))

Читаем identity и объединяем с client_profile...
Сохранено: /content/drive/MyDrive/ieee-db/train_identity_profiles.csv


,TransactionID,id_01,id_02,id_03,id_04,id_05,id_06,id_07,id_08,id_09,id_10,id_11,id_12,id_13,id_14,id_15,id_16,id_17,id_18,id_19,id_20,id_21,id_22,id_23,id_24,id_25,id_26,id_27,id_28,id_29,id_30,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo,client_id,tx_count_total,tx_amount_sum,tx_amount_mean,tx_amount_std,tx_count_fraud,fraud_rate,productcd_nunique,addr1_nunique,addr2_nunique,card4_mode,card6_mode,tx_dt_span_days
0,2987004,0.0,70787.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.0,NotFound,NaN,-480.0,New,NotFound,166.0,NaN,542.0,144.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,New,NotFound,Android 7.0,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M,4497_420.0_1,1.0,50.000,50.000000,0.000000,0.0,0.0,1.0,1.0,1.0,mastercard,credit,0.000000
1,2987008,-5.0,98945.0,NaN,NaN,0.0,-5.0,NaN,NaN,NaN,NaN,100.0,NotFound,49.0,-300.0,New,NotFound,166.0,NaN,621.0,500.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,New,NotFound,iOS 11.1.2,mobile safari 11.0,32.0,1334x750,match_status:1,T,F,F,T,mobile,iOS Device,2803_337.0_1,20.0,1080.450,54.022500,12.996764,0.0,0.0,3.0,1.0,1.0,visa,debit,126.201250
2,2987010,-5.0,191631.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,0.0,100.0,NotFound,52.0,NaN,Found,Found,121.0,NaN,410.0,142.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Found,Found,NaN,chrome 62.0,NaN,NaN,NaN,F,F,T,T,desktop,Windows,16496__0,4.0,235.396,58.849000,24.930425,0.0,0.0,1.0,1.0,0.0,mastercard,credit,31.031412
3,2987011,-5.0,221832.0,NaN,NaN,0.0,-6.0,NaN,NaN,NaN,NaN,100.0,NotFound,52.0,NaN,New,NotFound,225.0,NaN,176.0,507.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,New,NotFound,NaN,chrome 62.0,NaN,NaN,NaN,F,F,T,T,desktop,NaN,4461__1,16.0,410.439,25.652438,19.898303,0.0,0.0,1.0,1.0,0.0,mastercard,debit,39.639363
4,2987016,0.0,7460.0,0.0,0.0,1.0,0.0,NaN,NaN,0.0,0.0,100.0,NotFound,NaN,-300.0,Found,Found,166.0,15.0,529.0,575.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Found,Found,Mac OS X 10_11_6,chrome 62.0,24.0,1280x800,match_status:2,T,F,T,T,desktop,MacOS,1790_170.0_1,1.0,30.000,30.000000,0.000000,0.0,0.0,1.0,1.0,1.0,visa,debit,0.000000


In [10]:
# Сортируем транзакции по client_id и времени операции.
print("Сортируем транзакции по client_id и TransactionDT (пользуем chunked подход, если файл большой)...")
# Для большого файла используется чанковая обработка.
try:
    tx_full = pd.read_csv(tx_path)
    tx_full = ensure_cols(tx_full)
    for c in ['D1','addr1','TransactionDT','card1']:
        if c not in tx_full.columns:
            tx_full[c] = np.nan
    tx_full['D1n'] = np.floor(pd.to_numeric(tx_full['TransactionDT'], errors='coerce') / (24*3600)) - pd.to_numeric(tx_full['D1'], errors='coerce')
    tx_full['D1n'] = tx_full['D1n'].fillna(-9999).astype(int)
    tx_full['addr1'] = tx_full['addr1'].fillna('').astype(str)
    tx_full['client_id'] = tx_full['card1'].astype(str) + '_' + tx_full['addr1'] + '_' + tx_full['D1n'].astype(str)
    tx_sorted = tx_full.sort_values(["client_id","TransactionDT","TransactionID"], ascending=[True,True,True])
    out_sorted = base_path / "train_transaction_sorted_by_client_date.csv"
    tx_sorted.to_csv(out_sorted, index=False)
    print("Сохранено:", out_sorted)
    display(tx_sorted.head(20))
except Exception as e:
    print("Файл слишком большой для загрузки целиком, использую chunked external approach. Ошибка:", e)
    chunk_reader = pd.read_csv(tx_path, chunksize=200000)
    tmp_out = base_path / "train_transaction_with_clientid.csv"
    first = True
    for ch in chunk_reader:
        ch = ensure_cols(ch)
        for c in ['D1','addr1','TransactionDT','card1']:
            if c not in ch.columns:
                ch[c] = np.nan
        ch['D1n'] = np.floor(pd.to_numeric(ch['TransactionDT'], errors='coerce') / (24*3600)) - pd.to_numeric(ch['D1'], errors='coerce')
        ch['D1n'] = ch['D1n'].fillna(-9999).astype(int)
        ch['addr1'] = ch['addr1'].fillna('').astype(str)
        ch['client_id'] = ch['card1'].astype(str) + '_' + ch['addr1'] + '_' + ch['D1n'].astype(str)
        if first:
            ch.to_csv(tmp_out, index=False, mode="w")
            first = False
        else:
            ch.to_csv(tmp_out, index=False, header=False, mode="a")
    print("Добавлен столбец client_id и сохранён:", tmp_out)

Сортируем транзакции по client_id и TransactionDT (пользуем chunked подход, если файл большой)...
Сохранено: /content/drive/MyDrive/ieee-db/train_transaction_sorted_by_client_date.csv


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,addr1,addr2,dist1,dist2,P_emaildomain,R_emaildomain,C1,C2,C3,C4,C5,C6,C7,C8,C9,C10,C11,C12,C13,C14,D1,D2,D3,D4,D5,D6,D7,D8,D9,D10,D11,D12,D13,D14,D15,M1,M2,M3,M4,M5,M6,M7,M8,M9,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,V29,V30,V31,V32,V33,V34,V35,V36,V37,V38,V39,V40,V41,V42,V43,V44,V45,V46,V47,V48,V49,V50,V51,V52,V53,V54,V55,V56,V57,V58,V59,V60,V61,V62,V63,V64,V65,V66,V67,V68,V69,V70,V71,V72,V73,V74,V75,V76,V77,V78,V79,V80,V81,V82,V83,V84,V85,V86,V87,V88,V89,V90,V91,V92,V93,V94,V95,V96,V97,V98,V99,V100,V101,V102,V103,V104,V105,V106,V107,V108,V109,V110,V111,V112,V113,V114,V115,V116,V117,V118,V119,V120,V121,V122,V123,V124,V125,V126,V127,V128,V129,V130,V131,V132,V133,V134,V135,V136,V137,V138,V139,V140,V141,V142,V143,V144,V145,V146,V147,V148,V149,V150,V151,V152,V153,V154,V155,V156,V157,V158,V159,V160,V161,V162,V163,V164,V165,V166,V167,V168,V169,V170,V171,V172,V173,V174,V175,V176,V177,V178,V179,V180,V181,V182,V183,V184,V185,V186,V187,V188,V189,V190,V191,V192,V193,V194,V195,V196,V197,V198,V199,V200,V201,V202,V203,V204,V205,V206,V207,V208,V209,V210,V211,V212,V213,V214,V215,V216,V217,V218,V219,V220,V221,V222,V223,V224,V225,V226,V227,V228,V229,V230,V231,V232,V233,V234,V235,V236,V237,V238,V239,V240,V241,V242,V243,V244,V245,V246,V247,V248,V249,V250,V251,V252,V253,V254,V255,V256,V257,V258,V259,V260,V261,V262,V263,V264,V265,V266,V267,V268,V269,V270,V271,V272,V273,V274,V275,V276,V277,V278,V279,V280,V281,V282,V283,V284,V285,V286,V287,V288,V289,V290,V291,V292,V293,V294,V295,V296,V297,V298,V299,V300,V301,V302,V303,V304,V305,V306,V307,V308,V309,V310,V311,V312,V313,V314,V315,V316,V317,V318,V319,V320,V321,V322,V323,V324,V325,V326,V327,V328,V329,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339,D1n,client_id
146553,3169988,0,4050851,29.000,W,10000,111.0,150.0,mastercard,117.0,debit,184.0,87.0,NaN,NaN,gmail.com,NaN,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,4.0,1.0,83.0,83.0,70.0,70.0,70.0,NaN,NaN,NaN,NaN,83.0,NaN,NaN,NaN,NaN,83.0,NaN,NaN,NaN,M0,F,F,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.000000,0.000000,0.000000,0.0000,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-37,10000_184.0_-37
251453,3301550,0,7840676,42.777,C,10003,NaN,NaN,NaN,NaN,NaN,,NaN,NaN,NaN,gmail.com,gmail.com,1.0,1.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,NaN,NaN,0.0,NaN,0.0,NaN,NaN,NaN,0.0,NaN,0.0,0.0,0.0,0.0,NaN,NaN,NaN,M2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,

Готово — добавлены ячейки для загрузки данных, агрегации профилей, встраивания в `train_identity` и сортировки транзакций.  
Если хотите — могу добавить дополнительные профильные признаки (`geo_diversity`, `amount_repeat_share`, и т.п.) или ускорить сортировку большими файлами (external merge sort / dask / parquet).

**Расширённые признаки (in/out, медиана, activity, repeat/odd shares, MCC_risk, geo_diversity, identity-агрегаты)**

Следующие ячейки рассчитывают прокси-признаки и агрегаты identity, затем встраивают их в `client_profile` и сохраняют расширённые CSV-файлы.

In [11]:
# Расширенная фиче-инженерия: proxy-потоки, медиана, активность и риск-каналы.
# Признаки считаются векторно через groupby по slim-таблице.

high_risk_products = {"C", "H", "S"}
incoming_products = {"W", "R"}
needed_cols = [
    "client_id", "card1", "addr1", "D1", "TransactionDT", "TransactionAmt",
    "ProductCD", "card6", "P_emaildomain", "R_emaildomain"
]

print("Читаю slim-таблицу транзакций для быстрых groupby-агрегаций...")
tx_ext = pd.read_csv(tx_path, usecols=lambda c: c in needed_cols)
tx_ext = ensure_cols(tx_ext)

for c in needed_cols:
    if c not in tx_ext.columns:
        tx_ext[c] = np.nan

# Нормализуем ключ клиента
if 'client_id' in tx_ext.columns and not tx_ext['client_id'].isna().all():
    tx_ext['client_id'] = tx_ext['client_id'].astype(str)
else:
    tx_ext['TransactionDT'] = pd.to_numeric(tx_ext['TransactionDT'], errors='coerce')
    tx_ext['D1'] = pd.to_numeric(tx_ext['D1'], errors='coerce')
    tx_ext['addr1'] = tx_ext['addr1'].fillna('').astype(str)
    tx_ext['D1n'] = np.floor(tx_ext['TransactionDT'] / (24 * 3600)) - tx_ext['D1']
    tx_ext['D1n'] = tx_ext['D1n'].fillna(-9999).astype('int32')
    tx_ext['client_id'] = tx_ext['card1'].astype(str) + '_' + tx_ext['addr1'] + '_' + tx_ext['D1n'].astype(str)

tx_ext['TransactionDT'] = pd.to_numeric(tx_ext['TransactionDT'], errors='coerce')
tx_ext['TransactionAmt'] = pd.to_numeric(tx_ext['TransactionAmt'], errors='coerce').fillna(0.0)
tx_ext['ProductCD'] = tx_ext['ProductCD'].fillna('').astype(str)
tx_ext['addr1'] = tx_ext['addr1'].fillna('').astype(str)

# Базовый профиль используется как lookup, чтобы избежать дорогих .loc по каждому клиенту.
if 'client_profile' in globals() and 'client_id' in client_profile.columns:
    profile_base = client_profile.drop_duplicates('client_id', keep='last').set_index('client_id')
else:
    profile_base = pd.DataFrame(index=pd.Index([], name='client_id'))

card6_low = tx_ext['card6'].fillna('').astype(str).str.lower()
is_debit = card6_low.str.contains('debit', regex=False)
is_credit = card6_low.str.contains('credit', regex=False)
unknown_direction = ~(is_debit | is_credit)
is_incoming = is_credit | (unknown_direction & tx_ext['ProductCD'].isin(incoming_products))
is_outgoing = is_debit
is_high_risk_flow = tx_ext['ProductCD'].isin(high_risk_products)
is_standart_flow = ~is_high_risk_flow

tx_ext['tx_count_incoming_proxy'] = is_incoming.astype('int8')
tx_ext['tx_count_outgoing_proxy'] = is_outgoing.astype('int8')
tx_ext['tx_sum_incoming_proxy'] = tx_ext['TransactionAmt'].where(is_incoming, 0.0)
tx_ext['tx_sum_outgoing_proxy'] = tx_ext['TransactionAmt'].where(is_outgoing, 0.0)
tx_ext['tx_count_high_risk_flow'] = is_high_risk_flow.astype('int8')
tx_ext['tx_count_standart_flow'] = is_standart_flow.astype('int8')
tx_ext['tx_sum_high_risk_flow_proxy'] = tx_ext['TransactionAmt'].where(is_high_risk_flow, 0.0)
tx_ext['tx_sum_stanadart_flow_proxy_proxy'] = tx_ext['TransactionAmt'].where(is_standart_flow, 0.0)
tx_ext['odd_amount_flag'] = (np.abs(tx_ext['TransactionAmt'] - np.round(tx_ext['TransactionAmt'])) > 1e-6).astype('int8')
tx_ext['email_domain'] = tx_ext['P_emaildomain'].where(tx_ext['P_emaildomain'].notna(), tx_ext['R_emaildomain'])
tx_ext['tx_day'] = np.floor(tx_ext['TransactionDT'] / (24 * 3600))

print("Сортирую slim-таблицу и считаю интервалы между транзакциями...")
tx_ext = tx_ext.sort_values(['client_id', 'TransactionDT'], kind='mergesort')
tx_ext['dt_diff'] = tx_ext.groupby('client_id', sort=False)['TransactionDT'].diff()
tx_ext['dt_short_flag'] = tx_ext['dt_diff'].lt(24 * 3600).fillna(False).astype('int8')

print("Считаю агрегаты по клиентам через groupby...")
extended_df = tx_ext.groupby('client_id', sort=False).agg(
    tx_count_incoming_proxy=('tx_count_incoming_proxy', 'sum'),
    tx_count_outgoing_proxy=('tx_count_outgoing_proxy', 'sum'),
    tx_sum_incoming_proxy=('tx_sum_incoming_proxy', 'sum'),
    tx_sum_outgoing_proxy=('tx_sum_outgoing_proxy', 'sum'),
    tx_count_standart_flow=('tx_count_standart_flow', 'sum'),
    tx_count_high_risk_flow=('tx_count_high_risk_flow', 'sum'),
    tx_sum_stanadart_flow_proxy_proxy=('tx_sum_stanadart_flow_proxy_proxy', 'sum'),
    tx_sum_high_risk_flow_proxy=('tx_sum_high_risk_flow_proxy', 'sum'),
    tx_amount_median_proxy=('TransactionAmt', 'median'),
    odd_amount_count=('odd_amount_flag', 'sum'),
    dt_diff_sum=('dt_diff', 'sum'),
    dt_diff_cnt=('dt_diff', 'count'),
    dt_short_cnt=('dt_short_flag', 'sum'),
    amount_obs=('TransactionAmt', 'size'),
    amount_sum=('TransactionAmt', 'sum'),
    active_days=('tx_day', 'nunique'),
    dt_min=('TransactionDT', 'min'),
    dt_max=('TransactionDT', 'max')
)

print("Считаю точный repeat-share по суммам...")
amount_freq = (
    tx_ext.groupby(['client_id', 'TransactionAmt'], sort=False)
    .size()
    .rename('amount_freq')
    .reset_index()
)
repeated_amount_tx = (
    amount_freq.loc[amount_freq['amount_freq'] > 1]
    .groupby('client_id', sort=False)['amount_freq']
    .sum()
    .rename('repeated_amount_tx')
)
extended_df = extended_df.join(repeated_amount_tx, how='left')
extended_df['repeated_amount_tx'] = extended_df['repeated_amount_tx'].fillna(0.0)

extended_df['amount_repeat_share'] = extended_df['repeated_amount_tx'] / extended_df['amount_obs'].clip(lower=1)
extended_df['odd_amount_share'] = extended_df['odd_amount_count'] / extended_df['amount_obs'].clip(lower=1)
extended_df['avg_inter_tx_seconds'] = np.where(
    extended_df['dt_diff_cnt'] > 0,
    extended_df['dt_diff_sum'] / extended_df['dt_diff_cnt'],
    np.nan
)
extended_df['short_turnover_share'] = np.where(
    extended_df['dt_diff_cnt'] > 0,
    extended_df['dt_short_cnt'] / extended_df['dt_diff_cnt'],
    0.0
)
extended_df['MCC_risk_share_proxy'] = extended_df['tx_sum_high_risk_flow_proxy'] / (extended_df['amount_sum'] + 1e-9)

inout_total = extended_df['tx_count_incoming_proxy'] + extended_df['tx_count_outgoing_proxy']
extended_df['cash_out_ratio_proxy'] = np.where(inout_total > 0, extended_df['tx_count_outgoing_proxy'] / inout_total, 0.0)
extended_df['weighted_amount_repeat_share'] = extended_df['amount_repeat_share'] * np.minimum(1.0, extended_df['amount_obs'] / 15.0)

span_days_current = (extended_df['dt_max'] - extended_df['dt_min']) / (24 * 3600)

def profile_series(col_name, default=np.nan):
    if col_name in profile_base.columns:
        return pd.to_numeric(profile_base[col_name], errors='coerce').reindex(extended_df.index)
    return pd.Series(default, index=extended_df.index, dtype='float64')

tx_count_base = profile_series('tx_count_total')
mean_amt_base = profile_series('tx_amount_mean')
span_days_base = profile_series('tx_dt_span_days')
fraud_count_base = profile_series('tx_count_fraud', default=0.0)

tx_count_for_ratios = tx_count_base.fillna(extended_df['amount_obs'].astype(float))
span_days_for_freq = span_days_base.fillna(span_days_current)
safe_span_days = span_days_for_freq.clip(lower=1.0)
observed_days = np.floor(span_days_for_freq.clip(lower=0.0)) + 1.0

extended_df['tx_freq_per_day'] = tx_count_for_ratios / safe_span_days
extended_df['daily_activity_share'] = np.where(observed_days > 0, extended_df['active_days'] / observed_days, 0.0)
extended_df['daily_activity_share'] = extended_df['daily_activity_share'].clip(lower=0.0, upper=1.0)

mean_amt_current = extended_df['amount_sum'] / extended_df['amount_obs'].clip(lower=1)
mean_amt_for_ratios = mean_amt_base.fillna(mean_amt_current)
extended_df['high_risk_vs_mean'] = extended_df['tx_sum_high_risk_flow_proxy'] / (mean_amt_for_ratios + 1e-9)
extended_df['crypto_pattern_score'] = extended_df['MCC_risk_share_proxy'] * (mean_amt_for_ratios / (1.0 + mean_amt_for_ratios))
extended_df['profile_fraud_label'] = (fraud_count_base.fillna(0) > 0).astype('int8')

# Вспомогательные признаки по разреженной истории: помогают отличать реальный ноль от малого числа наблюдений.
extended_df['low_history_flag'] = (tx_count_for_ratios < 5).astype('int8')
extended_df['history_support_score'] = np.minimum(1.0, tx_count_for_ratios / 10.0)

print("Определяю наиболее частый email domain по клиенту...")
email_counts = (
    tx_ext.loc[tx_ext['email_domain'].notna(), ['client_id', 'email_domain']]
    .groupby(['client_id', 'email_domain'], sort=False)
    .size()
    .rename('email_count')
    .reset_index()
)
if not email_counts.empty:
    top_email = (
        email_counts.sort_values(['client_id', 'email_count', 'email_domain'], ascending=[True, False, True])
        .drop_duplicates('client_id')
        .set_index('client_id')[['email_domain']]
        .rename(columns={'email_domain': 'top_email_domain'})
    )
    extended_df = extended_df.join(top_email, how='left')
else:
    extended_df['top_email_domain'] = np.nan

# Оставляем только итоговые поля для стабильного merge.
extended_df = extended_df[[
    'tx_count_incoming_proxy',
    'tx_count_outgoing_proxy',
    'tx_sum_incoming_proxy',
    'tx_sum_outgoing_proxy',
    'tx_count_standart_flow',
    'tx_count_high_risk_flow',
    'tx_sum_stanadart_flow_proxy_proxy',
    'tx_sum_high_risk_flow_proxy',
    'tx_amount_median_proxy',
    'tx_freq_per_day',
    'daily_activity_share',
    'amount_repeat_share',
    'odd_amount_share',
    'avg_inter_tx_seconds',
    'short_turnover_share',
    'MCC_risk_share_proxy',
    'cash_out_ratio_proxy',
    'weighted_amount_repeat_share',
    'high_risk_vs_mean',
    'crypto_pattern_score',
    'low_history_flag',
    'history_support_score',
    'top_email_domain',
    'profile_fraud_label'
]].reset_index()

print("Расширенные признаки посчитаны для клиентов:", extended_df.shape[0])

# При повторном запуске удаляем пересекающиеся колонки, чтобы не было суффиксов _x/_y.
if 'client_profile' in globals():
    overlap_cols = [c for c in extended_df.columns if c != 'client_id' and c in client_profile.columns]
    if overlap_cols:
        client_profile = client_profile.drop(columns=overlap_cols)
    client_profile = client_profile.merge(extended_df, on='client_id', how='left')
else:
    client_profile = extended_df.copy()

out_profile = base_path / 'client_profile_extended.csv'
client_profile.to_csv(out_profile, index=False)
print('Сохранён расширенный client_profile ->', out_profile)



????? slim-??????? ?????????? ??? ??????? groupby-?????????...
???????? slim-??????? ? ?????? ????????? ????? ????????????...
?????? ???????? ?? ???????? ????? groupby...
?????? exact repeat-share ?? ??????...
????????? ???????? ?????? email domain ?? ???????...
Extended features computed for clients: 174280
???????? ??????????? client_profile -> /content/drive/MyDrive/ieee-db/client_profile_extended.csv


In [12]:
# Identity-агрегаты: числовые id_01..id_11 и категориальные id_12..id_38.
print("Собираю агрегаты по identity и встраиваю в client_profile...")

# Читаем identity, если таблица еще не загружена.
if 'identity' not in globals():
    identity = pd.read_csv(id_path)

if 'tx_map' not in globals():
    tx_map = pd.read_csv(tx_path, usecols=lambda c: c in ["TransactionID","card1","addr1","TransactionDT","D1","P_emaildomain","R_emaildomain"]) if 'tx_path' in globals() else pd.read_csv(str(base_path / 'train_transaction.csv'), usecols=lambda c: c in ["TransactionID","card1","addr1","TransactionDT","D1","P_emaildomain","R_emaildomain"])
    tx_map = ensure_cols(tx_map)
    for c in ['TransactionDT','D1','card1','addr1']:
        if c not in tx_map.columns:
            tx_map[c] = np.nan
    try:
        tx_map['D1n'] = np.floor(pd.to_numeric(tx_map['TransactionDT'], errors='coerce') / (24*3600)) - pd.to_numeric(tx_map['D1'], errors='coerce')
    except Exception:
        tx_map['D1n'] = np.nan
    tx_map['D1n'] = tx_map['D1n'].fillna(-9999).astype(int)
    tx_map['addr1'] = tx_map['addr1'].fillna('').astype(str)
    tx_map['client_id'] = tx_map['card1'].astype(str) + '_' + tx_map['addr1'] + '_' + tx_map['D1n'].astype(str)

identity = identity.merge(tx_map[['TransactionID','client_id']], on='TransactionID', how='left')

# Числовые id-колонки 01..11.
num_cols = [c for c in identity.columns if c.startswith('id_') and c[3:5].isdigit() and 1 <= int(c[3:5]) <= 11]
cat_cols = [c for c in identity.columns if c.startswith('id_') and c[3:5].isdigit() and 12 <= int(c[3:5]) <= 38]

num_agg = identity.groupby('client_id')[num_cols].agg(['mean','median','count']) if num_cols else pd.DataFrame({'client_id': identity['client_id'].unique()})
if not num_agg.empty:
    num_agg.columns = ['_'.join(col).strip() for col in num_agg.columns.values]
    num_agg = num_agg.reset_index()

# Моды категориальных id-колонок.
modes = []
for col in cat_cols:
    md = identity.groupby('client_id')[col].agg(lambda x: x.mode().iat[0] if not x.mode().empty else np.nan).reset_index().rename(columns={col: f"{col}_mode"})
    modes.append(md)

if modes:
    cat_agg = modes[0]
    for m in modes[1:]:
        cat_agg = cat_agg.merge(m, on='client_id', how='outer')
else:
    cat_agg = pd.DataFrame({'client_id': identity['client_id'].unique()})

identity_agg = num_agg.merge(cat_agg, on='client_id', how='outer') if not num_agg.empty else cat_agg

dt_cons = identity.groupby('client_id')['DeviceType'].nunique().reset_index().rename(columns={'DeviceType':'device_type_nunique'})
identity_agg = identity_agg.merge(dt_cons, on='client_id', how='left')

client_profile = client_profile.merge(identity_agg, on='client_id', how='left')

out_identity_profile = base_path / 'train_identity_profiles_extended.csv'
identity_agg.to_csv(out_identity_profile, index=False)
print('Сохранён identity_agg ->', out_identity_profile)

out_profile_ext = base_path / 'client_profile_extended.csv'
client_profile.to_csv(out_profile_ext, index=False)
print('Перезаписан/сохранён client_profile_extended ->', out_profile_ext)


Собираю агрегаты по identity и встраиваю в client_profile...
Сохранён identity_agg -> /content/drive/MyDrive/ieee-db/train_identity_profiles_extended.csv
Перезаписан/сохранён client_profile_extended -> /content/drive/MyDrive/ieee-db/client_profile_extended.csv


In [13]:
# Финализируем identity-признаки и сохраняем client_profile.
print('Вычисляю identity_present и num_missing_identity, формирую финальную схему колонок...')

# id_* колонки
id_cols = [c for c in identity.columns if c.startswith('id_')]
if len(id_cols) == 0:
    print('Предупреждение: не найдено id_* колонок в identity')

# Считаем строки identity и непустые id-значения по клиенту.
identity_rows = identity.groupby('client_id').size().reset_index(name='identity_rows')
if id_cols:
    non_null_per_col = identity.groupby('client_id')[id_cols].apply(lambda df: df.notna().sum()).reset_index()
    # суммируем по колонкам
    non_null_per_col['non_null_id_values'] = non_null_per_col[id_cols].sum(axis=1)
    non_null_total = non_null_per_col[['client_id','non_null_id_values']]
else:
    non_null_total = pd.DataFrame({'client_id': [], 'non_null_id_values': []})

identity_stats = identity_rows.merge(non_null_total, on='client_id', how='left')
identity_stats['non_null_id_values'] = identity_stats['non_null_id_values'].fillna(0).astype(int)
identity_stats['total_id_values_possible'] = identity_stats['identity_rows'] * max(len(id_cols), 0)
identity_stats['num_missing_identity'] = (identity_stats['total_id_values_possible'] - identity_stats['non_null_id_values']).clip(lower=0)
identity_stats['identity_present'] = (identity_stats['identity_rows'] > 0).astype(int)

identity_stats = identity_stats[['client_id','identity_present','num_missing_identity','identity_rows','non_null_id_values']]

# Объединяем с identity_agg и client_profile.
try:
    identity_agg = identity_agg.merge(identity_stats, on='client_id', how='left')
except Exception:
    identity_agg = identity_stats.copy()

client_profile = client_profile.merge(identity_stats, on='client_id', how='left')

# Заполняем пропуски identity-признаков нулями.
for col in ['identity_present', 'num_missing_identity', 'identity_rows', 'non_null_id_values']:
    if col not in client_profile.columns:
        client_profile[col] = 0
    client_profile[col] = pd.to_numeric(client_profile[col], errors='coerce').fillna(0)

client_profile['identity_present'] = client_profile['identity_present'].astype(int)
client_profile['num_missing_identity'] = client_profile['num_missing_identity'].astype(int)
client_profile['identity_rows'] = client_profile['identity_rows'].astype(int)
client_profile['non_null_id_values'] = client_profile['non_null_id_values'].astype(int)

final_cols = [
    'client_id',
    'tx_count_total',
    'tx_amount_sum','tx_amount_mean','tx_amount_median_proxy','tx_amount_std',
    'tx_count_standart_flow','tx_count_high_risk_flow','tx_sum_stanadart_flow_proxy_proxy','tx_sum_high_risk_flow_proxy',
    'tx_freq_per_day','daily_activity_share','avg_inter_tx_seconds','short_turnover_share',
    'amount_repeat_share','odd_amount_share','weighted_amount_repeat_share','cash_out_ratio_proxy',
    'MCC_risk_share_proxy','high_risk_vs_mean','crypto_pattern_score',
    'low_history_flag','history_support_score',
    'productcd_nunique','addr2_nunique','card4_mode','card6_mode','tx_dt_span_days',
    'top_email_domain','profile_fraud_label',
    'identity_present','num_missing_identity','identity_rows','non_null_id_values','device_type_nunique'
]

# Добавляем все возможные id_* агрегаты (если присутствуют в client_profile)
for c in client_profile.columns:
    if c.startswith('id_') and c not in final_cols:
        final_cols.append(c)

# Убедимся, что колонки существуют
for c in final_cols:
    if c not in client_profile.columns:
        client_profile[c] = np.nan

client_profile = client_profile[final_cols]

# Сохраняем финальный профиль и identity-агрегации.
out_profile_final = base_path / 'client_profile_final.csv'
client_profile.to_csv(out_profile_final, index=False)
print('Сохранён финальный client_profile ->', out_profile_final)

out_identity_agg = base_path / 'train_identity_agg_final.csv'
identity_agg.to_csv(out_identity_agg, index=False)
print('Сохранён финальный identity_agg ->', out_identity_agg)



Вычисляю identity_present и num_missing_identity, формирую финальную схему колонок...
Сохранён финальный client_profile -> /content/drive/MyDrive/ieee-db/client_profile_final.csv
Сохранён финальный identity_agg -> /content/drive/MyDrive/ieee-db/train_identity_agg_final.csv


## Делаем перемешанный файл

In [14]:

input_file = base_path / "client_profile_final.csv"
output_file = base_path / "client_profile_final_shuffled.csv"

# 3. Проверяем наличие файла и выполняем перемешивание
if input_file.exists():
    print(f"📖 Читаю файл: {input_file}")
    
    # Загружаем
    df = pd.read_csv(input_file)
    
    # Перемешиваем строки (frac=1 — это 100% данных)
    df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)
    
    # Сохраняем обратно на Диск
    df_shuffled.to_csv(output_file, index=False)
    
    print(f" Готово! Перемешанный файл сохранен: {output_file}")
    print(f" Количество строк: {len(df_shuffled)}")
else:
    print(f" Ошибка: Файл {input_file} не найден. Проверьте путь!")

📖 Читаю файл: /content/drive/MyDrive/ieee-db/client_profile_final.csv
 Готово! Перемешанный файл сохранен: /content/drive/MyDrive/ieee-db/client_profile_final_shuffled.csv
 Количество строк: 174280
